In [ ]:
import os
import re
import time
import json
import numpy as np
import pandas as pd
from PIL import Image
import nltk
import tensorflow as tf
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

tf.random.set_seed(42)

# Text constants (same as 03_glove_rnn_text.ipynb)
STOPWORDS = set(stopwords.words('english'))
MAX_LEN   = 50
VOCAB_SIZE = 10000
EMBED_DIM  = 100

# Image constants (same as 03_cnn_mobilenet.ipynb)
IMG_SIZE = (128, 128)
IMG_ROOT = '../data/Images'

# Known best hyperparams from unimodal runs
LSTM_UNITS  = 32   # best from 03_glove_rnn_text.ipynb
DENSE_UNITS = 256  # best from 03_cnn_mobilenet.ipynb

## Build paired DataFrame

Same paired construction as the other fusion notebooks: join LabeledText.csv with image
paths on the numeric file ID (`1.txt` ↔ `1.jpg`). One row per tweet.

In [ ]:
folder_to_label = {'Negative': 'negative', 'Neutral': 'neutral', 'positive': 'positive'}
img_records = []
for folder in folder_to_label:
    folder_path = os.path.join(IMG_ROOT, folder)
    for fname in os.listdir(folder_path):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            fid = int(os.path.splitext(fname)[0])
            img_records.append({'file_id': fid, 'img_path': os.path.join(folder_path, fname)})
img_df = pd.DataFrame(img_records)

text_df = pd.read_csv('../data/LabeledText.csv', encoding='latin-1')
text_df['file_id'] = text_df['File Name'].str.replace('.txt', '', regex=False).astype(int)
text_df['label']   = text_df['LABEL'].str.lower().str.strip()

paired = text_df.merge(img_df, on='file_id')[['file_id', 'Caption', 'label', 'img_path']]
paired = paired.dropna(subset=['Caption', 'img_path']).reset_index(drop=True)

print(f'Paired tweets: {len(paired)}')

## Joint train / val / test split

Same split as all unimodal notebooks. The base models train on **`train` only**; the
**val set is the meta-classifier's training data** (genuinely held out from the base models);
**test** is touched once at the end.

In [ ]:
trainval_df, test_df = train_test_split(
    paired, test_size=0.2, random_state=42, stratify=paired['label']
)
train_df, val_df = train_test_split(
    trainval_df, test_size=0.2, random_state=42, stratify=trainval_df['label']
)

print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

le = LabelEncoder()
le.fit(paired['label'])
print('Classes:', le.classes_)

y_train = le.transform(train_df['label'])
y_val   = le.transform(val_df['label'])
y_test  = le.transform(test_df['label'])

## Text preprocessing

Identical pipeline to `03_glove_rnn_text.ipynb`. Tokenizer fit on the joint training set only.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return ' '.join(tokens)

train_text = train_df['Caption'].apply(clean_text)
val_text   = val_df['Caption'].apply(clean_text)
test_text  = test_df['Caption'].apply(clean_text)

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(train_text)

def encode(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

X_train_txt = encode(train_text)
X_val_txt   = encode(val_text)
X_test_txt  = encode(test_text)

print('Text encoded. Shape:', X_train_txt.shape)

## Load GloVe and build embedding matrix

In [ ]:
glove = {}
with open('../data/glove.6B.100d.txt', encoding='utf-8') as f:
    for line in f:
        parts = line.split()
        glove[parts[0]] = np.array(parts[1:], dtype='float32')
print(f'GloVe vocab: {len(glove):,}')

embedding_matrix = np.zeros((VOCAB_SIZE, EMBED_DIM))
hits = 0
for word, idx in tokenizer.word_index.items():
    if idx < VOCAB_SIZE and word in glove:
        embedding_matrix[idx] = glove[word]
        hits += 1
print(f'GloVe coverage: {hits} / {min(VOCAB_SIZE, len(tokenizer.word_index))}')

## Load all images into memory

MobileNetV2 `preprocess_input` (scales to [-1, 1]). Loaded once in paired-DataFrame
order so indices align with the text arrays and split masks.

In [ ]:
def load_image(path):
    img = Image.open(path).convert('RGB')
    img = img.resize(IMG_SIZE, Image.LANCZOS)
    arr = np.array(img, dtype='float32')
    return mobilenet_preprocess(arr)

print(f'Loading {len(paired)} images...')
X_all_img = np.array([load_image(p) for p in paired['img_path']])
print(f'Loaded. Shape: {X_all_img.shape}')

X_train_img = X_all_img[train_df.index]
X_val_img   = X_all_img[val_df.index]
X_test_img  = X_all_img[test_df.index]

print(f'Train img: {X_train_img.shape}  Val: {X_val_img.shape}  Test: {X_test_img.shape}')

## Train GloVe + LSTM text model on the training set

Trained on `train` only so the val set stays held out — val predictions become the
meta-classifier's training features. `LSTM_UNITS=32`.

In [ ]:
t0 = time.time()

text_model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, weights=[embedding_matrix], trainable=False),
    LSTM(LSTM_UNITS),
    Dense(3, activation='softmax')
])
text_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
text_model.fit(
    X_train_txt, y_train,
    epochs=150, batch_size=32,
    callbacks=[EarlyStopping(monitor='loss', patience=20, restore_best_weights=True)],
    verbose=1
)

text_probs_val  = text_model.predict(X_val_txt)
text_probs_test = text_model.predict(X_test_txt)
text_acc = accuracy_score(y_test, np.argmax(text_probs_test, axis=1))
print(f'\nText model accuracy on joint test set: {text_acc:.3f}')

## Train MobileNetV2 image model on the training set

Trained on `train` only; `validation_split=0.2` carves an internal slice from `train`
purely for EarlyStopping. `DENSE_UNITS=256`.

In [ ]:
base = MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
base.trainable = False

img_model = Sequential([
    base,
    GlobalAveragePooling2D(),
    Dense(DENSE_UNITS, activation='relu', kernel_regularizer=l2(1e-4)),
    Dropout(0.5),
    Dense(3, activation='softmax')
])
img_model.compile(optimizer=Adam(learning_rate=1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
img_model.fit(
    X_train_img, y_train,
    validation_split=0.2, epochs=100, batch_size=32,
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
    verbose=1
)

img_probs_val  = img_model.predict(X_val_img)
img_probs_test = img_model.predict(X_test_img)
img_acc = accuracy_score(y_test, np.argmax(img_probs_test, axis=1))
print(f'\nImage model accuracy on joint test set: {img_acc:.3f}')

## Stacking — logistic regression meta-classifier

Concatenate the two models' softmax outputs into a 6-dim feature vector per tweet:
`[text_neg, text_neu, text_pos, img_neg, img_neu, img_pos]`.

Fit a logistic regression on the **val** features (held out from the base models) and
evaluate on **test**. Unlike weighted fusion's single scalar α, this learns a full 6→3
mapping that can weight each modality's per-class probability independently.

In [ ]:
Z_val  = np.hstack([text_probs_val,  img_probs_val])
Z_test = np.hstack([text_probs_test, img_probs_test])

meta = LogisticRegression(max_iter=1000, multi_class='multinomial')
meta.fit(Z_val, y_val)

y_stack    = meta.predict(Z_test)
runtime    = time.time() - t0

stack_acc  = accuracy_score(y_test, y_stack)
report     = classification_report(y_test, y_stack, target_names=le.classes_, output_dict=True)

print('=== Joint test set comparison ===')
print(f'  Text only  (GloVe+LSTM):       {text_acc:.3f}')
print(f'  Image only (MobileNetV2):      {img_acc:.3f}')
print(f'  Stacking (LogReg meta):        {stack_acc:.3f}')
print()
print(classification_report(y_test, y_stack, target_names=le.classes_))

## Inspect learned meta-classifier weights

If the image modality carries no usable signal, the meta-classifier should assign
near-zero magnitude to the three image features relative to the text features.
This makes the "image adds nothing" conclusion concrete.

In [ ]:
feat_names = ['text_neg', 'text_neu', 'text_pos', 'img_neg', 'img_neu', 'img_pos']
coef_df = pd.DataFrame(meta.coef_, columns=feat_names, index=[f'class_{c}' for c in le.classes_])
print('Meta-classifier coefficients (rows = output class, cols = input feature):')
print(coef_df.round(3).to_string())
print()

text_weight = np.abs(meta.coef_[:, :3]).sum()
img_weight  = np.abs(meta.coef_[:, 3:]).sum()
print(f'Total |coef| on text features:  {text_weight:.3f}')
print(f'Total |coef| on image features: {img_weight:.3f}')
print(f'Text/image weight ratio:        {text_weight / img_weight:.2f}x')

## Save metadata

In [ ]:
meta_out = {
    'model': 'multimodal_stacking',
    'accuracy': report['accuracy'],
    'macro_f1': report['macro avg']['f1-score'],
    'negative_f1': report['negative']['f1-score'],
    'neutral_f1': report['neutral']['f1-score'],
    'positive_f1': report['positive']['f1-score'],
    'runtime_seconds': runtime,
    'text_acc_joint_test': text_acc,
    'img_acc_joint_test': img_acc,
    'text_coef_abs_sum': float(text_weight),
    'img_coef_abs_sum': float(img_weight),
    'fusion_strategy': 'stacking_logreg_meta_on_val',
    'text_model': 'GloVe+LSTM (units=32)',
    'image_model': 'MobileNetV2 (dense=256)'
}

with open('../models/both/json/stacking_meta.json', 'w') as f:
    json.dump(meta_out, f, indent=2)

print(f'Saved. Stacking acc={stack_acc:.3f}. Total runtime: {runtime:.0f}s ({runtime/60:.1f} min)')